<a href="https://colab.research.google.com/github/nisas221/Gitpage/blob/main/qwen%208B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div class="markdown-google-sans">

<a name="machine-learning-examples"></a>

### Contoh bagus

</div>

- <a href="https://docs.jaxstack.ai/en/latest/JAX_for_LLM_pretraining.html">Melatih model bahasa miniGPT dengan Stack AI JAX</a>
- <a href="https://github.com/google/tunix/blob/main/examples/qlora_gemma.ipynb">Fine-tuning LoRA/QLoRA untuk LLM menggunakan Tunix</a>
- <a href="https://keras.io/examples/keras_recipes/parameter_efficient_finetuning_of_gemma_with_lora_and_qlora/">Parameter-efficient fine-tuning &#40;PEFT&#41; Gemma dengan LoRA dan QLoRA</a>
- <a href="https://keras.io/keras_hub/guides/hugging_face_keras_integration/">Memuat Checkpoint Hugging Face Transformer</a>
- <a href="https://keras.io/guides/int8_quantization_in_keras/">Kuantisasi Bilangan Bulat 8-bit di Keras</a>
- <a href="https://keras.io/examples/keras_recipes/float8_training_and_inference_with_transformer/">Pelatihan dan inferensi Float8 dengan model Transformer sederhana</a>
- <a href="https://keras.io/keras_hub/guides/transformer_pretraining/">Melakukan pra-pelatihan Transformer dari awal dengan KerasHub</a>
- <a href="https://keras.io/examples/vision/mnist_convnet/">Convnet MNIST sederhana</a>
- <a href="https://keras.io/examples/vision/image_classification_from_scratch/">Klasifikasi gambar dari awal menggunakan Keras 3</a>
- <a href="https://keras.io/keras_hub/guides/classification_with_keras_hub/">Klasifikasi Gambar dengan KerasHub</a>


In [ ]:
!pip install -U transformers accelerate bitsandbytes

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
import torch

model_name = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print("Model siap!")

In [ ]:
chat_history = []

while True:
    user_input = input("Anda: ")

    if user_input.lower() == "exit":
        break

    # Hitung panjang pertanyaan
    user_length = len(user_input.split())

    # Atur token otomatis
    max_tokens = min(
        max(128, user_length * 8),
        512
    )

    chat_history.append({
        "role": "user",
        "content": user_input
    })

    text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.4,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )